# 🧪 测试：开源模型批量生成古风图

先用 FLUX.1-schnell 生成 2 张测试图，确认能跑通再批量生 25 张。

> 菜单栏 → **代码执行程序** → **更改运行时类型** → 选 **T4 GPU**

In [ ]:
# Cell 1: 安装依赖
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q diffusers transformers accelerate safetensors

import torch
print(f'CUDA可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
# Cell 2: 加载 FLUX.1-schnell（~8GB显存，速度快）
from diffusers import FluxPipeline

pipe = FluxPipeline.from_pretrained(
    "black-forest-labs/FLUX.1-schnell",
    torch_dtype=torch.bfloat16
)
pipe = pipe.to("cuda")
pipe.enable_model_cpu_offload()  # 省显存

print('✅ FLUX.1-schnell 加载成功')

In [ ]:
# Cell 3: 生成 2 张测试图
from PIL import Image
import os

prompts = [
    "Chinese ancient ink wash painting, autumn ancient town, golden ginkgo leaves on blue tile roofs, mossy stone walls, misty mountains in distance, warm golden sunlight, cinematic, 16:9, poetic, serene, traditional Chinese aesthetic",
    "Chinese traditional painting style, ancient courtyard with moon gate, bamboo grove, stone lantern with moss, engraved plaque 'Yanqing Men', rain-washed stone path, jade green and ochre tones, misty, elegant, 16:9"
]

os.makedirs('/content/test_output', exist_ok=True)

for i, prompt in enumerate(prompts):
    print(f'生成图{i+1}...')
    image = pipe(
        prompt=prompt,
        guidance_scale=0.0,
        num_inference_steps=4,
        width=1024,
        height=576,
        generator=torch.Generator("cuda").manual_seed(42)
    ).images[0]
    
    image.save(f'/content/test_output/test_{i+1}.png')
    print(f'  ✅ test_{i+1}.png 保存完成')

print()
print('=== 测试完成！=== ')
print('文件列表:')
for f in os.listdir('/content/test_output'):
    print(f'  {f}')

---

## 如果测试通过

两张图风格一致 → 说明 FLUX 可以稳定输出统一风格

下一步就是用同一个 `manual_seed(42)` 批量生成全部 25 张图。

如果 FLUX 跑不动（显存不够），可以把 `torch_dtype=torch.bfloat16` 改成 `torch.float8` 或用 SDXL-Turbo 替代。